In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text

# Criando a Conexão com o Banco de Dados PostgreSQL

* Guardando os dados necessários para se conectar ao banco PostgreSQL

In [ ]:
usuario = DB_USER
senha = DB_SENHA
host = DB_HOST
porta = DB_PORTA
banco = DB_BANCO

* Criação do engine - gerencia conexões com o banco

In [3]:
engine = create_engine(f'postgresql+psycopg2://{usuario}:{senha}@{host}:{porta}/{banco}')

* Fazendo uma função para que facilite o uso da query nas consultas

In [4]:
def query(sql, params=None):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

# Realizando Consultas

* Pesquise por filmes do gênero Ação que apresentam Classificação maior ou igual a 14 anos

In [5]:
filmes_amor_terror = query('''
WITH filmes AS(
    SELECT titulo, genero, classificacao
    FROM filmes.filmes
    WHERE genero = 'Ação' and classificacao IN ('14', '16', '18')
    )
SELECT *
FROM filmes
ORDER BY 3 DESC
LIMIT 10
''')
filmes_amor_terror

,titulo,genero,classificacao
0,Pulp Fiction,Ação,18
1,Clube da Luta,Ação,18
2,Matrix Reloaded,Ação,16
3,O Poderoso Chefão Versão Estendida,Ação,16
4,O Senhor dos Anéis II,Ação,14
5,A Origem Reloaded,Ação,14
6,Forrest Gump Versão Estendida,Ação,14


* Quantos usuários distintos assistiram a algum filme em 2023?

In [6]:
usuarios_distintos = query('''
WITH usuarios_distinto AS(
    SELECT DISTINCT id_usuario, COUNT(*) AS total_usuarios
    FROM filmes.visualizacoes
    WHERE data_visualizacao BETWEEN '2023-01-01' AND '2023-12-31'
    GROUP BY 1
)
SELECT usuarios.nome AS usuario, total_usuarios
FROM usuarios_distinto
JOIN filmes.usuarios ON usuarios.id_usuario = usuarios_distinto.id_usuario
ORDER BY 2 DESC
LIMIT 10
''')
usuarios_distintos

,usuario,total_usuarios
0,André Oliveira,7
1,Juliana Souza,7
2,Lucas Santos,7
3,Lucas Souza,7
4,André Santos,6
5,André Barbosa,6
6,Mariana Pereira,6
7,Mariana Oliveira,6
8,Fernanda Silva,6
9,Mariana Barbosa,6


* Top 5 filmes por total de minutos assistidos. Ignore visualizações com tempo nulo.

In [7]:
top_5_filmes = query('''
WITH visu_filmes AS(
    SELECT id_filme, SUM(tempo_assistido_min) AS total_min_assistido
    FROM filmes.visualizacoes
    WHERE tempo_assistido_min IS NOT NULL
    GROUP BY 1
)
SELECT filmes.titulo AS filme, total_min_assistido
FROM visu_filmes
JOIN filmes.filmes ON filmes.id_filme = visu_filmes.id_filme
ORDER BY 2 DESC
LIMIT 5
''')
top_5_filmes

,filme,total_min_assistido
0,Top Gun: Maverick Reloaded,1570
1,Top Gun: Maverick Reloaded,1505
2,A Origem Versão Estendida,1465
3,Gladiador Versão Estendida,1390
4,O Poderoso Chefão II,1346


* Qual é a média de nota por gênero, arredondada para 2 casas, mas só mostre gêneros com pelo menos 20 avaliações.

In [8]:
nota_genero = query('''
WITH nota_filme AS(
    SELECT id_filme, AVG(nota) AS media_nota, COUNT(*) AS total_avaliacoes
    FROM filmes.avaliacoes
    GROUP BY 1    
),
    nota_genero AS(
        SELECT filmes.genero AS genero, ROUND(AVG(media_nota), 2) AS media_nota, SUM(total_avaliacoes) AS total_avaliacoes
        FROM nota_filme
        JOIN filmes.filmes ON nota_filme.id_filme = filmes.id_filme
        GROUP BY 1
        HAVING SUM(total_avaliacoes) >= 20
    )
SELECT *
FROM nota_genero
ORDER BY 2 DESC
''')
nota_genero

,genero,media_nota,total_avaliacoes
0,Suspense,3.40,90.0
1,Comédia,3.18,107.0
2,Drama,2.97,114.0
3,Ação,2.94,91.0
4,Romance,2.90,102.0
5,Ficção Científica,2.78,96.0


* Quais usuários estão com assinatura ATIVA e estão em planos “Premium” ou “Básico”?

In [9]:
usuarios_ativos = query('''
WITH ativos AS(
    SELECT usuarios.nome AS usuario, id_plano
    FROM filmes.assinaturas
    JOIN filmes.usuarios ON usuarios.id_usuario = assinaturas.id_usuario
    WHERE assinaturas.data_cancelamento IS NULL
),
    planos AS(
        SELECT ativos.usuario AS usuario, planos.nome_plano AS plano
        FROM ativos
        JOIN filmes.planos ON planos.id_plano = ativos.id_plano
        WHERE planos.nome_plano IN ('Premium', 'Básico')
    )
SELECT *
FROM planos
ORDER BY 1
''')
usuarios_ativos

,usuario,plano
0,Aline Barbosa,Premium
1,Aline Ferreira,Premium
2,Aline Pereira,Premium
3,Aline Souza,Premium
4,André Almeida,Básico
...,...,...
125,Mateus Pereira,Básico
126,Mateus Pereira,Básico
127,Mateus Silva,Básico
128,Mateus Silva,Básico


* Liste os países com mais usuários cadastrados.

In [10]:
paises_usuarios = query('''
WITH contagem_usuarios AS(
    SELECT pais, COUNT(*) AS total_usuarios
    FROM filmes.usuarios
    GROUP BY 1
)
SELECT *
FROM contagem_usuarios
ORDER BY 2 DESC
''')
paises_usuarios

,pais,total_usuarios
0,Estados Unidos,47
1,Brasil,45
2,Portugal,40
3,Canadá,35
4,Argentina,33


* Para cada mês de 2023, quantas visualizações aconteceram?

In [11]:
mes_visu = query('''
WITH visu_mes AS(
    SELECT TO_CHAR(data_visualizacao, 'YYYY-MM') AS mes, COUNT(*) AS total_visu
    FROM filmes.visualizacoes
    WHERE data_visualizacao BETWEEN '2023-01-01' AND '2023-12-31'
    GROUP BY 1
)
SELECT *
FROM visu_mes
ORDER BY 1
''')
mes_visu

,mes,total_visu
0,2023-01,50
1,2023-02,50
2,2023-03,43
3,2023-04,44
4,2023-05,63
5,2023-06,51
6,2023-07,64
7,2023-08,46
8,2023-09,47
9,2023-10,63


* Crie um ranking de filmes por “taxa de avaliação”: (número de avaliações / número de visualizações) para filmes com pelo menos 15 visualizações.

In [12]:
ranking_filmes = query('''
WITH avaliacoes AS (
    SELECT id_filme, COUNT(*) AS ava
    FROM filmes.avaliacoes
    GROUP BY id_filme
),
visualizacoes AS (
    SELECT id_filme, COUNT(*) AS visu
    FROM filmes.visualizacoes
    GROUP BY id_filme
)
SELECT
    f.titulo AS filme,
    a.ava,
    v.visu,
    ROUND((a.ava::numeric / v.visu), 4) AS pontuacao
FROM avaliacoes a
JOIN filmes.filmes f ON f.id_filme = a.id_filme
JOIN visualizacoes v ON v.id_filme = f.id_filme
WHERE v.visu >= 15
ORDER BY pontuacao DESC;
''')
ranking_filmes

,filme,ava,visu,pontuacao
0,Top Gun: Maverick Reloaded,13,15,0.8667
1,Top Gun: Maverick Reloaded,9,15,0.6000
2,Duna Reloaded,8,15,0.5333
3,Gladiador Reloaded,8,17,0.4706
4,Star Wars Remake,7,15,0.4667
5,Forrest Gump II,7,15,0.4667
6,Forrest Gump,7,15,0.4667
7,Titanic Versão Estendida,6,16,0.3750
8,Gladiador Reloaded,6,16,0.3750
9,Top Gun: Maverick Remake,7,19,0.3684


* Mostre quantas assinaturas existem por plano (nome do plano + quantidade)

In [13]:
assinaturas_plano = query('''
WITH plano_assinatura AS(
	SELECT id_plano, COUNT(*) AS total_assinatura
	FROM filmes.assinaturas
	GROUP BY 1
)
SELECT planos.nome_plano AS "Plano", total_assinatura
FROM plano_assinatura
JOIN filmes.planos ON planos.id_plano = plano_assinatura.id_plano
ORDER BY 2 DESC;
''')
assinaturas_plano

,Plano,total_assinatura
0,Premium,84
1,Básico,83
2,Intermediário,83


* Mudando o final dos emails dos usuarios que terminam com '@email.com'

In [14]:
mudando_email = query('''
WITH mudando_email AS(
	SELECT nome,
			email,
			REPLACE(email, '@email.com', '@gmail.com') AS novo_email,
			pais
	FROM filmes.usuarios
	WHERE email ILIKE '%@email.com%'
)
SELECT *
FROM mudando_email
LIMIT 10;
''')
mudando_email

,nome,email,novo_email,pais
0,Mateus Silva,mateus.silva@email.com,mateus.silva@gmail.com,Portugal
1,Mariana Oliveira,mariana.oliveira@email.com,mariana.oliveira@gmail.com,Brasil
2,André Santos,andré.santos@email.com,andré.santos@gmail.com,Canadá
3,Lucas Silva,lucas.silva@email.com,lucas.silva@gmail.com,Brasil
4,Mariana Costa,mariana.costa@email.com,mariana.costa@gmail.com,Canadá
5,André Souza,andré.souza@email.com,andré.souza@gmail.com,Canadá
6,Mariana Almeida,mariana.almeida@email.com,mariana.almeida@gmail.com,Canadá
7,Lucas Oliveira,lucas.oliveira@email.com,lucas.oliveira@gmail.com,Estados Unidos
8,Gabriel Oliveira,gabriel.oliveira@email.com,gabriel.oliveira@gmail.com,Argentina
9,Fernanda Santos,fernanda.santos@email.com,fernanda.santos@gmail.com,Brasil


* Quais avaliações tiveram nota entre 3 e 5 (inclusive) e foram feitas entre duas datas (ex.: 2023-01-01 e 2023-12-31)?

In [15]:
notas_avaliacoes = query('''
WITH filtro_ava AS(
	SELECT id_usuario, id_filme, nota, data_avaliacao
	FROM filmes.avaliacoes
	WHERE nota BETWEEN 3 AND 5
		AND data_avaliacao BETWEEN '2023-01-01' AND '2023-12-31'
)
SELECT *
FROM filtro_ava
ORDER BY nota DESC
LIMIT 10;
''')
notas_avaliacoes

,id_usuario,id_filme,nota,data_avaliacao
0,69,53,5,2023-01-11
1,9,5,5,2023-12-23
2,159,32,5,2023-05-09
3,58,19,5,2023-01-08
4,39,58,5,2023-02-04
5,87,49,5,2023-08-08
6,145,64,5,2023-10-10
7,74,36,5,2023-03-28
8,28,74,5,2023-05-11
9,95,15,5,2023-12-09


* Quais filmes tiveram pelo menos 3 avaliações e qual a média de nota arredondada para 2 casas?

In [16]:
filmes_notas = query('''
WITH visu_geral AS(
	SELECT id_filme, ROUND(AVG(nota), 2) AS media_nota , COUNT(*) total_avaliacao
	FROM filmes.avaliacoes
	GROUP BY 1
),
	calculos AS(
		SELECT *
		FROM visu_geral
		WHERE total_avaliacao > 3
	)
SELECT *
FROM calculos
ORDER BY media_nota DESC
LIMIT 10;
''')
filmes_notas

,id_filme,media_nota,total_avaliacao
0,85,4.43,7
1,13,4.33,6
2,22,4.25,4
3,65,4.25,4
4,62,4.20,5
5,39,4.00,4
6,59,4.00,4
7,63,4.00,8
8,38,4.00,4
9,88,3.88,8


* Para cada usuário, qual foi o total de minutos assistidos (soma) e o máximo de minutos em uma única visualização?

In [17]:
usuarios_min = query('''
WITH visu_geral AS(
	SELECT id_usuario, tempo_assistido_min
	FROM filmes.visualizacoes
),
	calculos AS(
		SELECT id_usuario, SUM(tempo_assistido_min) AS total_min, MAX(tempo_assistido_min) AS tempo_max
		FROM visu_geral
		GROUP BY 1
		ORDER BY 2 DESC
	)
SELECT *
FROM calculos
LIMIT 10;
''')
usuarios_min

,id_usuario,total_min,tempo_max
0,199,1062,169
1,30,987,167
2,118,919,164
3,69,919,167
4,162,866,167
5,48,847,134
6,97,846,159
7,45,822,162
8,141,817,165
9,124,789,152


* Quais filmes tiveram visualizações em 2023?

In [18]:
visu_filmes = query('''
WITH base AS(
	SELECT filmes.titulo , visualizacoes.data_visualizacao
	FROM filmes.visualizacoes
	JOIN filmes.filmes ON filmes.id_filme = visualizacoes.id_filme
	WHERE visualizacoes.data_visualizacao BETWEEN '2023-01-01' AND '2023-12-31'
),
	calculos AS(
		SELECT titulo, TO_CHAR(data_visualizacao, 'MM/YYYY') AS mes_ano, COUNT(*) AS total_visu
		FROM base
		GROUP BY 1, 2
	)
SELECT * 
FROM calculos
ORDER BY mes_ano, total_visu DESC;
''')

* Para cada plano, mostre o preço, número de telas e uma classificação usando CASE:

In [19]:
preco_plano = query('''
WITH base AS(
	SELECT id_plano, nome_plano, preco_mensal
	FROM filmes.planos
),
	calculo AS(
		SELECT *,
		CASE
			WHEN preco_mensal < 20 THEN 'BARATO'
			WHEN preco_mensal >= 20 AND preco_mensal <45 THEN 'OK'
			ELSE 'CARO'
		END AS faixa_preco
		FROM base
	)
SELECT *
FROM calculo;
''')
preco_plano

,id_plano,nome_plano,preco_mensal,faixa_preco
0,1,Básico,19.9,BARATO
1,2,Intermediário,29.9,OK
2,3,Premium,49.9,CARO
